## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score rapidfuzz

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo (using importlib to handle hyphens in path)
try:
    import importlib.util
    
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Define task name for consistent file organization
    TASK_NAME = "location-extraction"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)

#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Machine learning and transformers
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    Trainer, 
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

# Evaluation metrics
import evaluate
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Data Loading and Preparation

### 2.1 Load Raw Data

In [ ]:
# Try loading from cloned repository first
try:
    data = pd.read_csv('/content/code-satp/data/location_info_augmented.csv', dtype=str)
    print("✅ Data loaded successfully from cloned repository")
    print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")
    
    # Fallback to GitHub if cloned repo fails or if working locally
    url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
    try:
        data = pd.read_csv(url, dtype=str)
        print("✅ Data loaded successfully from GitHub")
        print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
    except Exception as e:
        print(f"❌ Error loading CSV from GitHub: {e}")

### 2.2 Create Structured Location Labels

#### About Structured Output Format

This notebook trains a T5-base model to extract location information from incident summaries using a **structured output format** with explicit labels for each administrative level.

**Key Changes from Previous Approach:**

1. **Data Source**
   - Using `location_info_augmented.csv` which contains human-annotated and augmented location data
   - Ground truth comes from separate columns: `state`, `district`, `village_name`, `other_locations`
   - `other_locations` combines filtered blocks (only those in text) + other_areas (forests, police stations, etc.)
   - All duplicates between block/other_areas and village names have been removed

2. **Structured Output Format**
   - **Old Format:** `"state: Chhattisgarh, district: Sukma, block: Dornapal, village: Nilamadgu"` (block often missing)
   - **New Format:** `"state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"` (more robust)
   
   **Benefits:**
   - Clear hierarchy with labeled levels
   - `other_locations` provides flexible context (blocks, forests, police stations, etc.)
   - Better extractability: all values in `other_locations` appear in incident text
   - Expected F1 improvement: 70-75% (vs 41% for blocks)
   - Handles missing levels gracefully
   - Enables level-specific evaluation metrics

3. **Enhanced Evaluation Metrics**
   Instead of simple token matching, we now compute:
   - **Exact Match Accuracy**: All levels must match perfectly
   - **Per-Level Metrics**: Precision, Recall, F1 for each of state/district/village/other_locations
   - **Micro-averaged F1**: Overall performance across all levels
   - **Level-by-level breakdown**: See exactly which administrative levels the model struggles with

4. **Better for Downstream Tasks**
   The structured output can be directly:
   - Parsed into a dictionary for programmatic access
   - Fed into geocoding APIs with component filtering (state/district as filters, village+other_locations as address)
   - Used for geographic analysis at specific administrative levels
   - Validated against known location hierarchies

#### Build Structured Location Column

In [ ]:
# Create structured human_annotated_location column from individual location fields
def build_structured_location(row):
    """Build structured location string from state, district, village, other_locations columns."""
    parts = []
    
    if pd.notna(row.get('state')) and str(row.get('state')).strip():
        parts.append(f"state: {str(row['state']).strip()}")
    
    if pd.notna(row.get('district')) and str(row.get('district')).strip():
        parts.append(f"district: {str(row['district']).strip()}")
    
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip():
        parts.append(f"village: {str(row['village_name']).strip()}")
    
    if pd.notna(row.get('other_locations')) and str(row.get('other_locations')).strip():
        parts.append(f"other_locations: {str(row['other_locations']).strip()}")
    
    return ', '.join(parts) if parts else ''

# Apply to create the human_annotated_location column
data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

# Display sample to verify
print("Sample human-annotated locations:")
display(data[['state', 'district', 'village_name', 'other_locations', 'human_annotated_location']].head(10))

#### Verify Data Structure

In [ ]:
# Display the first few rows to verify the data structure
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

### 2.3 Data Preprocessing and Filtering

#### Create Train/Validation/Test Splits

In [ ]:
from datasets import Dataset, DatasetDict

def prepare_data(df):
    # Updated prompt with structured output format
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, village: <name>, other_locations: <name>. Use exact format with labels. Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['human_annotated_location'].tolist()
    # Preserve metadata columns (date) for later use in evaluation/geocoding
    # Convert datetime to string format to avoid PyArrow NaT issues
    dates = []
    for d in df['date']:
        if pd.isna(d):
            dates.append(None)
        else:
            # Convert to string format YYYY-MM-DD
            try:
                dates.append(pd.to_datetime(d).strftime('%Y-%m-%d'))
            except:
                dates.append(None)
    return {'input_text': inputs, 'target_text': targets, 'date': dates}

# TEMPORAL SPLIT: Sort by date for chronological train/val/test split
# This ensures clean separation of Telangana formation (June 2, 2014)
# Training on earlier data, testing on more recent data (realistic deployment scenario)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data = data.sort_values('date').reset_index(drop=True)

# Calculate split indices for 80/10/10 split
train_end_idx = int(len(data) * 0.8)
val_end_idx = int(len(data) * 0.9)

# Create temporal splits
train_data = data.iloc[:train_end_idx]
val_data = data.iloc[train_end_idx:val_end_idx]
test_data = data.iloc[val_end_idx:]

print(f"Temporal split summary:")
print(f"  Training:   {len(train_data)} incidents ({train_data['date'].min()} to {train_data['date'].max()})")
print(f"  Validation: {len(val_data)} incidents ({val_data['date'].min()} to {val_data['date'].max()})")
print(f"  Test:       {len(test_data)} incidents ({test_data['date'].min()} to {test_data['date'].max()})")
print(f"\nNote: Training ends before Telangana formation (June 2, 2014)")
print(f"      Test set contains only post-2014 incidents with current state names")

# Create datasets from temporal splits
dataset = DatasetDict({
    'train': Dataset.from_dict(prepare_data(train_data)),
    'validation': Dataset.from_dict(prepare_data(val_data)),
    'test': Dataset.from_dict(prepare_data(test_data))
})

print(f"\n{dataset}")


DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7854
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
})


## 3. Model Setup and Training

**Padding Token Workflow:**
1. **During preprocessing** (Section 3.1): Padding tokens in labels are replaced with `-100`
2. **During training**: The loss function automatically ignores tokens marked as `-100`
3. **During decoding** (Section 5+): Filter out `-100` tokens before decoding: `[l for l in label if l != -100]`

**Optimization:**
- Training optimizes **cross-entropy loss** (`eval_loss`) on validation set
- Task-specific metrics (exact match, per-level F1) are computed **after training** for evaluation
- BLEU/ROUGE metrics are not used (better suited for text generation than structured extraction)

### 3.1 Initialize Tokenizer and Preprocessing

In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Tokenization function
def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding='max_length'
    )

    # Tokenize targets and get input_ids
    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding='max_length'
    ).input_ids

    # Replace padding token id's in the labels with -100 so they are ignored by the loss
    labels = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels]

    model_inputs['labels'] = labels
    return model_inputs

# Apply tokenization to all splits
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print("Tokenization complete!")
print(f"Train set: {len(tokenized_datasets['train'])} examples")
print(f"Validation set: {len(tokenized_datasets['validation'])} examples")
print(f"Test set: {len(tokenized_datasets['test'])} examples")


### 3.2 GPU Configuration

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### 3.4 Initialize Model and Training Configuration

**Training Strategy:**
- **10 epochs maximum** with **early stopping** (patience=3 epochs)
- Training stops if validation loss doesn't improve for 3 consecutive epochs
- Prevents overfitting while allowing model to fully converge
- Saves only the 3 best checkpoints to conserve disk space
- Final model is the one with lowest validation loss

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and logs
    eval_strategy="epoch",           # Evaluate at the end of every epoch
    learning_rate=5e-5,              # Learning rate
    per_device_train_batch_size=8,   # Batch size for training
    per_device_eval_batch_size=4,    # Batch size for evaluation
    num_train_epochs=10,             # Maximum number of training epochs
    weight_decay=0.01,               # Weight decay for regularization
    save_strategy="epoch",           # Save the model at each epoch
    save_total_limit=3,              # Keep only 3 best checkpoints (saves disk space)
    logging_dir="./logs",            # Directory for storing logs
    logging_steps=100,               # Log every 100 steps
    load_best_model_at_end=True,     # Load the best model at the end of training
    metric_for_best_model="eval_loss",  # Metric to monitor for best model
    greater_is_better=False,         # Lower eval_loss is better
    report_to='none'
)

# Initialize the Trainer with early stopping callback
# Early stopping: stops training if eval_loss doesn't improve for 3 epochs
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    # compute_metrics=compute_metrics
)

### 3.5 Train the Model

In [ ]:

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.089200,0.073098
2,0.074200,0.066961
3,0.071400,0.064558


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2946, training_loss=0.13490951619662966, metrics={'train_runtime': 3414.9844, 'train_samples_per_second': 6.9, 'train_steps_per_second': 0.863, 'total_flos': 1.434826581737472e+16, 'train_loss': 0.13490951619662966, 'epoch': 3.0})

### 3.6 Save Trained Model

In [ ]:
# Save the model and tokenizer to standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / "t5base_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/spiece.model',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/added_tokens.json')

## 4. Evaluation Metrics Definition

This section defines the evaluation functions used to assess model performance on the temporal test set. These functions are used in Section 5 after loading the trained model.

### 4.1 Parse Structured Location Output

In [ ]:
def parse_structured_location(text):
    """Parse structured location string into a dictionary."""
    location_dict = {
        'state': None,
        'district': None,
        'village': None,
        'other_locations': None
    }
    
    if not text or text.strip() == '':
        return location_dict
    
    # Split by comma and parse each part
    parts = [part.strip() for part in text.split(',')]
    for part in parts:
        if ':' in part:
            label, value = part.split(':', 1)
            label = label.strip().lower()
            value = value.strip()
            if label in location_dict:
                location_dict[label] = value
    
    return location_dict

# Test the parser with a sample
sample_location = "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
parsed = parse_structured_location(sample_location)
print("Sample parsing test:")
print(f"Input: {sample_location}")
print(f"Parsed: {parsed}")

### 4.2 Dual Metrics: Exact vs Fuzzy Matching

We compute **two sets of metrics** to comprehensively evaluate location extraction:

1. **Exact Match Metrics** (strict)
   - Requires perfect string match (case-insensitive)
   - Example: `"Vishakhapatnam"` ≠ `"Visakhapatnam"` ❌
   - Best for: Understanding model's exact accuracy

2. **Fuzzy Match Metrics** (lenient, 85% similarity threshold)
   - Handles spelling variations using Levenshtein distance
   - Example: `"Vishakhapatnam"` ≈ `"Visakhapatnam"` ✅ (similarity: 93%)
   - Best for: Real-world usability where spelling variations are common

**Special Handling for `other_locations`:**
- Uses **token-level F1** (since it's multi-value: `"Dornapal, Chintagufa Forest"`)
- Order-independent: `"A, B"` = `"B, A"`
- Partial credit: Extracting 1 of 2 locations gives 50% recall
- Both exact and fuzzy matching applied at token level

**Why Both Metrics Matter:**
- **Exact F1**: Shows if model needs more training data or better architecture
- **Fuzzy F1**: Shows real-world performance for geocoding (where small variations don't matter)
- **Gap between them**: Indicates how much performance loss is due to spelling variations vs actual errors

In [ ]:
from rapidfuzz import fuzz

def fuzzy_match(str1, str2, threshold=85):
    """
    Check if two strings match with fuzzy matching (handles spelling variations).
    
    Returns True if:
    - Both strings are None (both fields are empty)
    - Both strings exist and have >= threshold similarity
    
    Returns False if:
    - Only one string is None (one field exists, the other doesn't)
    - Both strings exist but similarity < threshold
    """
    # Both None means both fields are empty - this is a match
    if str1 is None and str2 is None:
        return True
    # One is None but not the other - this is NOT a match
    if str1 is None or str2 is None:
        return False
    # Both exist - check fuzzy similarity
    return fuzz.ratio(str1.lower(), str2.lower()) >= threshold

def compute_detailed_metrics(predictions, labels):
    """
    Compute detailed metrics with both exact and fuzzy matching.
    
    Returns metrics at three levels:
    1. Full record accuracy: % of examples where ALL fields match
    2. Micro-averaged metrics: Aggregate P/R/F1 across all field extractions
       - Calculated by summing correct/predicted/actual across all fields
       - Gives equal weight to each field extraction (not each example)
    3. Field-level metrics: Individual P/R/F1 for state, district, village, other_locations
    
    Note: Fuzzy match accuracy (full record) is typically low because it requires
    ALL fields to fuzzy match. Field-level fuzzy metrics are more informative.
    """
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]
    
    # Initialize counters for EXACT matching
    exact_matches = 0
    exact_core_matches = 0  # state + district + village only
    exact_level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0},
        'other_locations': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    # Initialize counters for FUZZY matching
    fuzzy_matches = 0
    fuzzy_core_matches = 0  # state + district + village only
    fuzzy_level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0},
        'other_locations': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    total_examples = len(decoded_preds)
    
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_dict = parse_structured_location(pred)
        label_dict = parse_structured_location(label)
        
        # Check exact match for entire prediction (all 4 fields)
        if pred_dict == label_dict:
            exact_matches += 1
        
        # Check exact match for core geographic hierarchy (state + district + village only)
        core_exact_match = True
        for level in ['state', 'district', 'village']:
            if pred_dict[level] != label_dict[level]:
                core_exact_match = False
                break
        if core_exact_match:
            exact_core_matches += 1
        
        # Check fuzzy match for entire prediction (all 4 fields must fuzzy match)
        all_levels_fuzzy_match = True
        for level in ['state', 'district', 'village', 'other_locations']:
            if not fuzzy_match(pred_dict[level], label_dict[level]):
                all_levels_fuzzy_match = False
                break
        if all_levels_fuzzy_match:
            fuzzy_matches += 1
        
        # Check fuzzy match for core geographic hierarchy (state + district + village only)
        core_fuzzy_match = True
        for level in ['state', 'district', 'village']:
            if not fuzzy_match(pred_dict[level], label_dict[level]):
                core_fuzzy_match = False
                break
        if core_fuzzy_match:
            fuzzy_core_matches += 1
        
        # Compute per-level metrics (EXACT)
        for level in ['state', 'district', 'village']:
            if label_dict[level] is not None:
                exact_level_metrics[level]['total'] += 1
            if pred_dict[level] is not None:
                exact_level_metrics[level]['predicted'] += 1
            if pred_dict[level] is not None and label_dict[level] is not None:
                if pred_dict[level].lower() == label_dict[level].lower():
                    exact_level_metrics[level]['correct'] += 1
        
        # Compute per-level metrics (FUZZY)
        for level in ['state', 'district', 'village']:
            if label_dict[level] is not None:
                fuzzy_level_metrics[level]['total'] += 1
            if pred_dict[level] is not None:
                fuzzy_level_metrics[level]['predicted'] += 1
            if pred_dict[level] is not None and label_dict[level] is not None:
                if fuzzy_match(pred_dict[level], label_dict[level]):
                    fuzzy_level_metrics[level]['correct'] += 1
        
        # Special handling for other_locations (token-level F1) - EXACT
        if label_dict['other_locations'] is not None:
            label_tokens = set([t.strip().lower() for t in label_dict['other_locations'].split(',') if t.strip()])
            exact_level_metrics['other_locations']['total'] += len(label_tokens)
            
            if pred_dict['other_locations'] is not None:
                pred_tokens = set([t.strip().lower() for t in pred_dict['other_locations'].split(',') if t.strip()])
                exact_level_metrics['other_locations']['predicted'] += len(pred_tokens)
                exact_level_metrics['other_locations']['correct'] += len(pred_tokens & label_tokens)
            # If prediction is None but label exists, predicted stays 0
        elif pred_dict['other_locations'] is not None:
            # Prediction exists but label is None (false positives)
            pred_tokens = set([t.strip().lower() for t in pred_dict['other_locations'].split(',') if t.strip()])
            exact_level_metrics['other_locations']['predicted'] += len(pred_tokens)
        
        # Special handling for other_locations (token-level F1) - FUZZY
        if label_dict['other_locations'] is not None:
            label_tokens = [t.strip() for t in label_dict['other_locations'].split(',') if t.strip()]
            fuzzy_level_metrics['other_locations']['total'] += len(label_tokens)
            
            if pred_dict['other_locations'] is not None:
                pred_tokens = [t.strip() for t in pred_dict['other_locations'].split(',') if t.strip()]
                fuzzy_level_metrics['other_locations']['predicted'] += len(pred_tokens)
                
                # For fuzzy matching, count matches using fuzzy string similarity
                matched_pred = set()
                matched_label = set()
                for p_idx, p_token in enumerate(pred_tokens):
                    for l_idx, l_token in enumerate(label_tokens):
                        if l_idx not in matched_label and fuzzy_match(p_token, l_token):
                            fuzzy_level_metrics['other_locations']['correct'] += 1
                            matched_pred.add(p_idx)
                            matched_label.add(l_idx)
                            break
        elif pred_dict['other_locations'] is not None:
            # Prediction exists but label is None (false positives)
            pred_tokens = [t.strip() for t in pred_dict['other_locations'].split(',') if t.strip()]
            fuzzy_level_metrics['other_locations']['predicted'] += len(pred_tokens)
    
    # Compute precision, recall, F1 for each level (EXACT)
    results = {
        'exact_match': exact_matches / total_examples * 100,
        'exact_core_match': exact_core_matches / total_examples * 100,
        'total_examples': total_examples
    }
    
    for level in ['state', 'district', 'village', 'other_locations']:
        metrics = exact_level_metrics[level]
        precision = (metrics['correct'] / metrics['predicted'] * 100) if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total'] * 100) if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_exact_precision'] = precision
        results[f'{level}_exact_recall'] = recall
        results[f'{level}_exact_f1'] = f1
        results[f'{level}_support'] = metrics['total']
    
    # Compute precision, recall, F1 for each level (FUZZY)
    results['fuzzy_match'] = fuzzy_matches / total_examples * 100
    results['fuzzy_core_match'] = fuzzy_core_matches / total_examples * 100
    
    for level in ['state', 'district', 'village', 'other_locations']:
        metrics = fuzzy_level_metrics[level]
        precision = (metrics['correct'] / metrics['predicted'] * 100) if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total'] * 100) if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_fuzzy_precision'] = precision
        results[f'{level}_fuzzy_recall'] = recall
        results[f'{level}_fuzzy_f1'] = f1
    
    # Compute micro-averaged F1 (weighted by support) - EXACT
    total_correct = sum(exact_level_metrics[level]['correct'] for level in ['state', 'district', 'village', 'other_locations'])
    total_predicted = sum(exact_level_metrics[level]['predicted'] for level in ['state', 'district', 'village', 'other_locations'])
    total_actual = sum(exact_level_metrics[level]['total'] for level in ['state', 'district', 'village', 'other_locations'])
    
    micro_exact_precision = (total_correct / total_predicted * 100) if total_predicted > 0 else 0
    micro_exact_recall = (total_correct / total_actual * 100) if total_actual > 0 else 0
    micro_exact_f1 = (2 * micro_exact_precision * micro_exact_recall / (micro_exact_precision + micro_exact_recall)) if (micro_exact_precision + micro_exact_recall) > 0 else 0
    
    results['micro_exact_precision'] = micro_exact_precision
    results['micro_exact_recall'] = micro_exact_recall
    results['micro_exact_f1'] = micro_exact_f1
    
    # Compute micro-averaged F1 (weighted by support) - FUZZY
    total_correct = sum(fuzzy_level_metrics[level]['correct'] for level in ['state', 'district', 'village', 'other_locations'])
    total_predicted = sum(fuzzy_level_metrics[level]['predicted'] for level in ['state', 'district', 'village', 'other_locations'])
    total_actual = sum(fuzzy_level_metrics[level]['total'] for level in ['state', 'district', 'village', 'other_locations'])
    
    micro_fuzzy_precision = (total_correct / total_predicted * 100) if total_predicted > 0 else 0
    micro_fuzzy_recall = (total_correct / total_actual * 100) if total_actual > 0 else 0
    micro_fuzzy_f1 = (2 * micro_fuzzy_precision * micro_fuzzy_recall / (micro_fuzzy_precision + micro_fuzzy_recall)) if (micro_fuzzy_precision + micro_fuzzy_recall) > 0 else 0
    
    results['micro_fuzzy_precision'] = micro_fuzzy_precision
    results['micro_fuzzy_recall'] = micro_fuzzy_recall
    results['micro_fuzzy_f1'] = micro_fuzzy_f1
    
    # Print summary - formatted as a table
    print("="*80)
    print("EVALUATION RESULTS")
    print("="*80)
    print(f"Total Examples: {total_examples}")
    print()
    
    # Overall accuracy (full record matches)
    print(f"{'FULL RECORD ACCURACY':<40} {'Exact Match':<15} {'Fuzzy Match':<15}")
    print("-" * 70)
    print(f"{'All fields correct (incl. other_locations)':<40} {results['exact_match']:>6.2f}%        {results['fuzzy_match']:>6.2f}%")
    print(f"{'Core geography correct (state+district+village)':<40} {results['exact_core_match']:>6.2f}%        {results['fuzzy_core_match']:>6.2f}%")
    print()
    print("Note: 'Core geography' excludes other_locations field to focus on the main")
    print("      administrative hierarchy (state → district → village).")
    print()
    
    # Micro-averaged metrics (aggregated across all fields)
    print(f"{'MICRO-AVERAGED METRICS':<40} {'Exact Match':<15} {'Fuzzy Match':<15}")
    print("(Aggregated across all fields: state, district, village, other_locations)")
    print("-" * 70)
    print(f"{'Precision':<40} {micro_exact_precision:>6.2f}%        {micro_fuzzy_precision:>6.2f}%")
    print(f"{'Recall':<40} {micro_exact_recall:>6.2f}%        {micro_fuzzy_recall:>6.2f}%")
    print(f"{'F1 Score':<40} {micro_exact_f1:>6.2f}%        {micro_fuzzy_f1:>6.2f}%")
    print()
    
    # Field-level metrics table
    print(f"{'FIELD-LEVEL METRICS':<20} {'Match Type':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'Support':<10}")
    print("-" * 78)
    
    for level in ['state', 'district', 'village', 'other_locations']:
        level_display = level.replace('_', ' ').title()
        
        # Exact match row
        print(f"{level_display:<20} {'Exact':<12} "
              f"{results[f'{level}_exact_precision']:>6.2f}%     "
              f"{results[f'{level}_exact_recall']:>6.2f}%     "
              f"{results[f'{level}_exact_f1']:>6.2f}%     "
              f"{results[f'{level}_support']:>6.0f}")
        
        # Fuzzy match row
        print(f"{'':<20} {'Fuzzy':<12} "
              f"{results[f'{level}_fuzzy_precision']:>6.2f}%     "
              f"{results[f'{level}_fuzzy_recall']:>6.2f}%     "
              f"{results[f'{level}_fuzzy_f1']:>6.2f}%     "
              f"{'':<10}")
        print()
    
    print("="*80)
    print("\nKEY INSIGHTS:")
    print("- Micro-averaged metrics: Aggregate performance across all field extractions")
    print("- Field-level metrics: Individual performance for each location component")
    print("- Support: Number of examples in test set with this field present")
    print("- Fuzzy matching: Allows for minor spelling variations (85% similarity threshold)")
    print("="*80)
    
    return results

## 5. Model Evaluation on Test Set

**Workflow:**
- Load a previously trained model (from Section 3 or a saved checkpoint)
- Evaluate it on the temporal test set (from Section 2) using metrics defined in Section 4
- Generate predictions and save detailed results

**Important:** This section requires:
- Section 2 to be run first (to create the temporal split)
- Section 4 to be run first (to define evaluation functions)
- Either Section 3 (to train a new model) or a pre-existing saved model

### 5.1 Load Model and Tokenizer

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer from standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_path = results_dir / "t5base_finetuned_model"
    print(f"Loading model from: {model_path}")
except NameError:
    # Fallback to Drive path
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    print(f"Loading model from: {model_path}")

model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'✅ Model loaded successfully')
print(f'Using device: {device}')

Using device: cuda
Using device: cuda


### 5.2 Prepare Test Dataset for Inference

**Note:** This section uses the temporal test dataset created in Section 2.3. Make sure to run cells in order (from Section 2 onwards) to ensure the `dataset` variable with the temporal split is available.

In [ ]:
import pandas as pd

# Use the temporal test dataset that was created in Section 2.3
# Note: Requires running cells in order (Section 2.3 must be run first)
print(f"Using temporal test dataset created in Section 2.3")
print(f"Dataset size: {len(dataset['test'])} incidents")
print(f"Features: {', '.join(dataset['test'].features.keys())}")

# Display sample
print("\nSample from test set:")
for i in range(min(3, len(dataset['test']))):
    print(f"\nExample {i+1}:")
    print(f"  Input:  {dataset['test'][i]['input_text'][:100]}...")
    print(f"  Target: {dataset['test'][i]['target_text']}")
    if 'date' in dataset['test'][i]:
        print(f"  Date:   {dataset['test'][i]['date']}")

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization to the temporal test set
tokenized_dataset_test = dataset['test'].map(preprocess_function, batched=True)

# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print(f"\n✅ Test dataset tokenized and ready for inference")


Map:   0%|          | 0/982 [00:00<?, ? examples/s]

In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute detailed metrics with breakdown
    metrics = compute_detailed_metrics(all_predictions, all_labels)
    return metrics

### 5.3 Run Evaluation on Test Set

Uses the evaluation functions defined in Section 4 to compute comprehensive metrics on the temporal test set.

**Understanding the Evaluation Metrics:**

1. **Full Record Accuracy**: Percentage of examples where fields match
   - **All fields correct**: All 4 fields (state, district, village, other_locations) must match
   - **Core geography correct**: Only state + district + village must match (excludes other_locations)
   - Exact Match: Character-perfect matching
   - Fuzzy Match: 85% similarity threshold (allows minor spelling variations)

2. **Micro-Averaged Metrics**: Aggregated performance across all field extractions
   - Sums correct/predicted/actual counts across all 4 fields
   - Gives equal weight to each field extraction (not each example)
   - Better indicator of overall extraction quality than full record accuracy

3. **Field-Level Metrics**: Individual Precision/Recall/F1 for each location component
   - State, District, Village: Direct string matching
   - Other Locations: Token-level matching (handles comma-separated lists)
   - Most granular view of model performance

In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'exact_match': 40.52953156822811}


### 5.4 Save Evaluation Metrics

In [ ]:
# Save evaluation metrics to CSV
metrics_df = pd.DataFrame([metrics])
metrics_df['model'] = 'T5-base'
metrics_df['dataset'] = 'test'
metrics_df['task'] = 'location-extraction'

# Reorder columns for clarity - both exact and fuzzy metrics
cols_order = [
    'task', 'model', 'dataset', 
    'exact_match', 'exact_core_match', 'fuzzy_match', 'fuzzy_core_match',
    'micro_exact_f1', 'micro_exact_precision', 'micro_exact_recall',
    'micro_fuzzy_f1', 'micro_fuzzy_precision', 'micro_fuzzy_recall',
    'state_exact_f1', 'state_exact_precision', 'state_exact_recall',
    'state_fuzzy_f1', 'state_fuzzy_precision', 'state_fuzzy_recall',
    'district_exact_f1', 'district_exact_precision', 'district_exact_recall',
    'district_fuzzy_f1', 'district_fuzzy_precision', 'district_fuzzy_recall',
    'village_exact_f1', 'village_exact_precision', 'village_exact_recall',
    'village_fuzzy_f1', 'village_fuzzy_precision', 'village_fuzzy_recall',
    'other_locations_exact_f1', 'other_locations_exact_precision', 'other_locations_exact_recall',
    'other_locations_fuzzy_f1', 'other_locations_fuzzy_precision', 'other_locations_fuzzy_recall',
    'state_support', 'district_support', 'village_support', 'other_locations_support',
    'total_examples'
]
metrics_df = metrics_df[[col for col in cols_order if col in metrics_df.columns]]

# Save using the standard utility function
try:
    saved_path = save_dataframe_csv(metrics_df, "location_extraction_test_metrics.csv", task_name=TASK_NAME)
    print(f"✅ Evaluation metrics saved to: {saved_path}")
except NameError:
    # Fallback if file_io utils not available
    output_path = "./location_extraction_test_metrics.csv"
    metrics_df.to_csv(output_path, index=False)
    print(f"✅ Evaluation metrics saved to: {output_path}")

display(metrics_df)

### 5.5 Generate and Save Test Predictions

In [ ]:
# Generate and save predictions for the entire test set
print("Generating predictions for full test set...")

model.eval()
all_predictions = []
all_labels = []
all_inputs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        # Generate predictions
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=64
        )
        
        # Decode predictions and inputs
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        inputs = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        
        # Decode labels: filter out -100 (padding tokens ignored during training)
        truths = [
            tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
            for label in labels
        ]
        
        all_predictions.extend(preds)
        all_labels.extend(truths)
        all_inputs.extend(inputs)

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'input_text': all_inputs,
    'ground_truth': all_labels,
    'prediction': all_predictions
})

# Add date column from test dataset for Telangana state correction in geocoding
predictions_df['date'] = dataset['test']['date']

# Parse structured outputs
predictions_df['gt_state'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['gt_district'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['gt_other_locations'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('other_locations'))
predictions_df['gt_village'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('village'))

predictions_df['pred_state'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['pred_district'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['pred_other_locations'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('other_locations'))
predictions_df['pred_village'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('village'))

# Add match indicators
predictions_df['exact_match'] = predictions_df.apply(
    lambda row: parse_structured_location(row['prediction']) == parse_structured_location(row['ground_truth']), 
    axis=1
)
predictions_df['state_match'] = predictions_df.apply(
    lambda row: (row['pred_state'] or '').lower() == (row['gt_state'] or '').lower() if row['gt_state'] else pd.NA,
    axis=1
)
predictions_df['district_match'] = predictions_df.apply(
    lambda row: (row['pred_district'] or '').lower() == (row['gt_district'] or '').lower() if row['gt_district'] else pd.NA,
    axis=1
)
predictions_df['other_locations_match'] = predictions_df.apply(
    lambda row: (row['pred_other_locations'] or '').lower() == (row['gt_other_locations'] or '').lower() if row['gt_other_locations'] else pd.NA,
    axis=1
)
predictions_df['village_match'] = predictions_df.apply(
    lambda row: (row['pred_village'] or '').lower() == (row['gt_village'] or '').lower() if row['gt_village'] else pd.NA,
    axis=1
)

# Save predictions
try:
    saved_path = save_dataframe_csv(predictions_df, "location_extraction_test_predictions.csv", task_name=TASK_NAME)
    print(f"✅ Test predictions saved to: {saved_path}")
    print(f"   Total predictions: {len(predictions_df)}")
    print(f"   Exact matches: {predictions_df['exact_match'].sum()} ({predictions_df['exact_match'].sum()/len(predictions_df)*100:.1f}%)")
except NameError:
    output_path = "./location_extraction_test_predictions.csv"
    predictions_df.to_csv(output_path, index=False)
    print(f"✅ Test predictions saved to: {output_path}")

display(predictions_df.head(10))

### 5.6 Verify Saved Results

In [ ]:
# List all files in the results directory
try:
    results_dir = get_task_results_dir(TASK_NAME)
    print(f"Results directory: {results_dir}\n")
    print("Saved files:")
    print("\n".join(sorted(os.listdir(results_dir))))
except NameError:
    print("Results saved in current directory:")
    import glob
    csv_files = glob.glob("location_extraction*.csv")
    if csv_files:
        print("\n".join(sorted(csv_files)))
    else:
        print("No result files found yet.")

In [ ]:
# Quick view: Show 10 random examples with side-by-side comparison
model.eval()
batch = next(iter(test_loader))

input_ids = batch["input_ids"][:10].to(device)
attention_mask = batch["attention_mask"][:10].to(device)
labels = batch["labels"][:10]

# Generate predictions
with torch.no_grad():
    outputs = model.generate(input_ids, attention_mask=attention_mask, max_length=64)

# Decode
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
ground_truth = [
    tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
    for label in labels
]

# Display side by side with structured parsing
print("\nSample Predictions vs Ground Truth (Structured Format):")
print("="*100)
for i, (pred, truth) in enumerate(zip(predictions, ground_truth), 1):
    pred_dict = parse_structured_location(pred)
    truth_dict = parse_structured_location(truth)
    match = "✓ EXACT MATCH" if pred_dict == truth_dict else "✗ PARTIAL/NO MATCH"
    
    print(f"\n[{i}] {match}")
    print(f"Predicted:    '{pred}'")
    print(f"Ground Truth: '{truth}'")
    
    if pred_dict != truth_dict:
        # Show level-by-level comparison
        print("  Level breakdown:")
        for level in ['state', 'district', 'village', 'other_locations']:
            pred_val = pred_dict[level] or "—"
            truth_val = truth_dict[level] or "—"
            if pred_val == truth_val:
                print(f"    {level:16}: ✓ {truth_val}")
            else:
                print(f"    {level:16}: ✗ predicted='{pred_val}', actual='{truth_val}'")

## 6. Geocoding Evaluation: Google Maps API

This section evaluates the **geolocation accuracy** of our T5 model's predictions by:
1. Loading the predictions saved from Section 5
2. Converting model predictions to lat/long coordinates using Google Maps API
3. Comparing against the manual baseline (human-looked-up coordinates)
4. Computing distance-based metrics to assess practical utility

### 6.1 Setup: Load Libraries and API Key

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# Google Maps API key - stored securely in Colab Secrets
# Add your key: Click 🔑 icon → Add secret → Name: 'googlemapsAPI' → Paste key → Enable notebook access
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

print("✅ Google Maps API configured")
print(f"   API key loaded: {API_KEY[:10]}..." if API_KEY else "⚠️  No API key found - add to Colab Secrets")

Using device: cuda
Using device: cuda


### 6.2 Load Saved Predictions

In [ ]:
# Load the predictions saved in Section 5.5
try:
    results_dir = get_task_results_dir(TASK_NAME)
    predictions_path = f"{results_dir}/location_extraction_test_predictions.csv"
    test_predictions = pd.read_csv(predictions_path)
    print(f"✅ Loaded test predictions from: {predictions_path}")
except (NameError, FileNotFoundError):
    # Fallback: try loading from current directory
    test_predictions = pd.read_csv("./location_extraction_test_predictions.csv")
    print("✅ Loaded test predictions from current directory")

print(f"\nDataset size: {len(test_predictions)} incidents")
print(f"Columns: {', '.join(test_predictions.columns)}")
print(f"\nSample predictions:")
display(test_predictions[['prediction', 'ground_truth']].head(3))

### 6.3 Define Geocoding Functions

#### 6.3.1 Get Location Details

In [ ]:
def get_location_details(address, components=None):
    """
    Get location details using Google Geocoding API with optional component filtering.
    
    Args:
        address: Address string (village + other_locations)
        components: Optional dict with 'state' and/or 'district' for filtering
        
    Returns:
        Dict with state, district, subdistrict, town_village, latitude, longitude
    """
    # Build component filter string
    component_parts = ['country:IN']  # Always filter by India
    
    if components:
        if components.get('state'):
            component_parts.append(f"administrative_area_level_1:{components['state']}")
        if components.get('district'):
            component_parts.append(f"administrative_area_level_2:{components['district']}")
    
    params = {
        'address': address,
        'key': API_KEY,
        'components': '|'.join(component_parts)
    }
    
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        # Don't print error for ZERO_RESULTS (expected for some locations)
        if data['status'] != 'ZERO_RESULTS':
            print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

print("✅ Geocoding function defined")

#### 6.3.2 Parse Structured to Address and Components

In [ ]:
def parse_structured_to_address_and_components(structured_location):
    """
    Parse T5 structured output and build address + components for Google API.
    
    Strategy with fallbacks:
    1. Prefer village + other_locations as address (most specific)
    2. Fall back to district if no village/other_locations (district centroid)
    3. Fall back to state if only state available (state centroid)
    4. Use appropriate component filters based on what's in the address
    
    Examples:
    Input: "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
    Output: {'address': 'Nilamadgu, Dornapal', 'components': {'state': 'Chhattisgarh', 'district': 'Sukma'}}
    
    Input: "state: Chhattisgarh, district: Sukma" (no village)
    Output: {'address': 'Sukma', 'components': {'state': 'Chhattisgarh'}}
    
    Input: "state: Chhattisgarh" (only state)
    Output: {'address': 'Chhattisgarh', 'components': None}
    """
    if not structured_location or structured_location.strip() == '':
        return None
    
    parsed = parse_structured_location(structured_location)
    
    # Build address from most specific to least specific
    address_parts = []
    
    # Level 1: Village + other_locations (most specific)
    if parsed['village']:
        address_parts.append(parsed['village'])
    if parsed['other_locations']:
        address_parts.append(parsed['other_locations'])
    
    # Level 2: Fallback to district if no village/other_locations
    if not address_parts and parsed['district']:
        address_parts.append(parsed['district'])
    
    # Level 3: Fallback to state if nothing else available
    if not address_parts and parsed['state']:
        address_parts.append(parsed['state'])
    
    # Build component filters (don't filter by what's already in the address)
    components = {}
    
    # If we're using village/other_locations as address, use state+district as filters
    if parsed['village'] or parsed['other_locations']:
        if parsed['state']:
            components['state'] = parsed['state']
        if parsed['district']:
            components['district'] = parsed['district']
    
    # If we're using district as address, only use state as filter
    elif parsed['district'] and address_parts and parsed['district'] in address_parts[0]:
        if parsed['state']:
            components['state'] = parsed['state']
    
    # If we're using state as address, no additional filters needed
    # (Google will understand "Chhattisgarh, India" without extra components)
    
    # If we have nothing to geocode, return None
    if not address_parts:
        return None
    
    return {
        'address': ', '.join(address_parts),
        'components': components if components else None
    }

print("✅ Parser function defined (with district/state fallback)")

### 6.4 Geocode Model Predictions

In [ ]:
# Geocode the T5 model predictions
print(f"Geocoding {len(test_predictions)} model predictions...")
print("This will make 2 API calls per row (model prediction + human baseline)")
print(f"Expected API calls: ~{len(test_predictions) * 2}")
print("="*80)

geocoding_results = []

for idx, row in test_predictions.iterrows():
    # Parse the model's prediction
    parsed = parse_structured_to_address_and_components(row['prediction'])
    
    result = {
        'index': idx,
        'input_text': row['input_text'],
        'model_prediction': row['prediction'],
        'ground_truth': row['ground_truth'],
    }
    
    # Geocode model prediction
    if parsed:
        location_details = get_location_details(parsed['address'], parsed['components'])
        if location_details:
            result['model_lat'] = location_details['latitude']
            result['model_lon'] = location_details['longitude']
            result['model_geocoded_state'] = location_details['state']
            result['model_geocoded_district'] = location_details['district']
        else:
            result['model_lat'] = None
            result['model_lon'] = None
            result['model_geocoded_state'] = None
            result['model_geocoded_district'] = None
    else:
        result['model_lat'] = None
        result['model_lon'] = None
        result['model_geocoded_state'] = None
        result['model_geocoded_district'] = None
    
    # Geocode ground truth (human baseline)
    parsed_gt = parse_structured_to_address_and_components(row['ground_truth'])
    
    if parsed_gt:
        location_details_gt = get_location_details(parsed_gt['address'], parsed_gt['components'])
        if location_details_gt:
            result['human_lat'] = location_details_gt['latitude']
            result['human_lon'] = location_details_gt['longitude']
            result['human_geocoded_state'] = location_details_gt['state']
            result['human_geocoded_district'] = location_details_gt['district']
        else:
            result['human_lat'] = None
            result['human_lon'] = None
            result['human_geocoded_state'] = None
            result['human_geocoded_district'] = None
    else:
        result['human_lat'] = None
        result['human_lon'] = None
        result['human_geocoded_state'] = None
        result['human_geocoded_district'] = None
    
    geocoding_results.append(result)
    
    # Progress indicator
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(test_predictions)} rows...")

geocoded_df = pd.DataFrame(geocoding_results)
print(f"\n✅ Geocoding complete!")
print(f"   Model predictions geocoded: {geocoded_df['model_lat'].notna().sum()}/{len(geocoded_df)}")
print(f"   Human baseline geocoded: {geocoded_df['human_lat'].notna().sum()}/{len(geocoded_df)}")

display(geocoded_df[['model_prediction', 'model_lat', 'model_lon', 'human_lat', 'human_lon']].head(10))

### 6.5 Calculate Distance Metrics

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points on Earth (in km).
    """
    if any(x is None for x in [lat1, lon1, lat2, lon2]):
        return None
    
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    # Earth's radius in kilometers
    R = 6371.0
    
    return R * c

# Calculate distances
geocoded_df['distance_km'] = geocoded_df.apply(
    lambda row: haversine_distance(
        row['model_lat'], row['model_lon'],
        row['human_lat'], row['human_lon']
    ),
    axis=1
)

# Filter to only rows where both model and human were successfully geocoded
valid_comparisons = geocoded_df[geocoded_df['distance_km'].notna()].copy()

print(f"Valid comparisons (both model and human geocoded): {len(valid_comparisons)}/{len(geocoded_df)}")
print(f"\nDistance Statistics:")
print(f"  Mean distance: {valid_comparisons['distance_km'].mean():.2f} km")
print(f"  Median distance: {valid_comparisons['distance_km'].median():.2f} km")
print(f"  Min distance: {valid_comparisons['distance_km'].min():.2f} km")
print(f"  Max distance: {valid_comparisons['distance_km'].max():.2f} km")
print(f"  Std deviation: {valid_comparisons['distance_km'].std():.2f} km")

# Distance buckets for practical interpretation
print(f"\nDistance Distribution:")
print(f"  Within 10 km:   {(valid_comparisons['distance_km'] <= 10).sum()} ({(valid_comparisons['distance_km'] <= 10).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Within 50 km:   {(valid_comparisons['distance_km'] <= 50).sum()} ({(valid_comparisons['distance_km'] <= 50).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Within 100 km:  {(valid_comparisons['distance_km'] <= 100).sum()} ({(valid_comparisons['distance_km'] <= 100).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Beyond 100 km:  {(valid_comparisons['distance_km'] > 100).sum()} ({(valid_comparisons['distance_km'] > 100).sum()/len(valid_comparisons)*100:.1f}%)")

display(valid_comparisons[['model_prediction', 'ground_truth', 'distance_km']].head(10))

### 6.5.1 Analyze Geocoding Failures

In [ ]:
# Analyze why geocoding failed for some cases
print("="*80)
print("GEOCODING FAILURE ANALYSIS")
print("="*80)

# Breakdown of failures
model_failed = geocoded_df['model_lat'].isna().sum()
human_failed = geocoded_df['human_lat'].isna().sum()
both_succeeded = (geocoded_df['model_lat'].notna() & geocoded_df['human_lat'].notna()).sum()
both_failed = (geocoded_df['model_lat'].isna() & geocoded_df['human_lat'].isna()).sum()
only_model_failed = (geocoded_df['model_lat'].isna() & geocoded_df['human_lat'].notna()).sum()
only_human_failed = (geocoded_df['model_lat'].notna() & geocoded_df['human_lat'].isna()).sum()

print(f"\nTotal test cases: {len(geocoded_df)}")
print(f"\nGeocoding Success Rates:")
print(f"  Both succeeded:        {both_succeeded} ({both_succeeded/len(geocoded_df)*100:.1f}%)")
print(f"  Both failed:           {both_failed} ({both_failed/len(geocoded_df)*100:.1f}%)")
print(f"  Only model failed:     {only_model_failed} ({only_model_failed/len(geocoded_df)*100:.1f}%)")
print(f"  Only human failed:     {only_human_failed} ({only_human_failed/len(geocoded_df)*100:.1f}%)")

print(f"\nModel geocoding:  {len(geocoded_df) - model_failed}/{len(geocoded_df)} succeeded ({(len(geocoded_df) - model_failed)/len(geocoded_df)*100:.1f}%)")
print(f"Human geocoding:  {len(geocoded_df) - human_failed}/{len(geocoded_df)} succeeded ({(len(geocoded_df) - human_failed)/len(geocoded_df)*100:.1f}%)")

# Look at examples where both failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE BOTH MODEL AND HUMAN FAILED TO GEOCODE")
print("="*80)
both_failed_df = geocoded_df[(geocoded_df['model_lat'].isna()) & (geocoded_df['human_lat'].isna())]
print(f"\nTotal cases where both failed: {len(both_failed_df)}")
if len(both_failed_df) > 0:
    print("\nSample cases:")
    display(both_failed_df[['model_prediction', 'ground_truth']].head(10))

# Look at examples where only model failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE ONLY MODEL FAILED (HUMAN SUCCEEDED)")
print("="*80)
only_model_failed_df = geocoded_df[(geocoded_df['model_lat'].isna()) & (geocoded_df['human_lat'].notna())]
print(f"\nTotal cases: {len(only_model_failed_df)}")
if len(only_model_failed_df) > 0:
    print("\nSample cases (model extraction may be incomplete/wrong):")
    display(only_model_failed_df[['model_prediction', 'ground_truth']].head(10))

# Look at examples where only human failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE ONLY HUMAN FAILED (MODEL SUCCEEDED)")
print("="*80)
only_human_failed_df = geocoded_df[(geocoded_df['model_lat'].notna()) & (geocoded_df['human_lat'].isna())]
print(f"\nTotal cases: {len(only_human_failed_df)}")
if len(only_human_failed_df) > 0:
    print("\nSample cases (human annotation may have issues):")
    display(only_human_failed_df[['model_prediction', 'ground_truth']].head(10))

### 6.6 Save Geocoding Results

In [ ]:
# Save the full geocoded results
try:
    saved_path = save_dataframe_csv(geocoded_df, "location_extraction_geocoding_results.csv", task_name=TASK_NAME)
    print(f"✅ Geocoding results saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_results.csv"
    geocoded_df.to_csv(output_path, index=False)
    print(f"✅ Geocoding results saved to: {output_path}")

# Save summary metrics
summary_metrics = {
    'total_test_cases': len(geocoded_df),
    'model_geocoded_successfully': geocoded_df['model_lat'].notna().sum(),
    'human_geocoded_successfully': geocoded_df['human_lat'].notna().sum(),
    'valid_comparisons': len(valid_comparisons),
    'mean_distance_km': valid_comparisons['distance_km'].mean() if len(valid_comparisons) > 0 else None,
    'median_distance_km': valid_comparisons['distance_km'].median() if len(valid_comparisons) > 0 else None,
    'within_10km': (valid_comparisons['distance_km'] <= 10).sum() if len(valid_comparisons) > 0 else 0,
    'within_50km': (valid_comparisons['distance_km'] <= 50).sum() if len(valid_comparisons) > 0 else 0,
    'within_100km': (valid_comparisons['distance_km'] <= 100).sum() if len(valid_comparisons) > 0 else 0,
    'beyond_100km': (valid_comparisons['distance_km'] > 100).sum() if len(valid_comparisons) > 0 else 0,
}

summary_df = pd.DataFrame([summary_metrics])

try:
    saved_path = save_dataframe_csv(summary_df, "location_extraction_geocoding_metrics.csv", task_name=TASK_NAME)
    print(f"✅ Geocoding metrics saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_metrics.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"✅ Geocoding metrics saved to: {output_path}")

display(summary_df)

### 6.7 Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot distance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid_comparisons['distance_km'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Distances: Model vs Human Baseline')
axes[0].axvline(valid_comparisons['distance_km'].median(), color='red', linestyle='--', 
                label=f"Median: {valid_comparisons['distance_km'].median():.1f} km")
axes[0].legend()

# Cumulative distribution
sorted_distances = np.sort(valid_comparisons['distance_km'])
cumulative = np.arange(1, len(sorted_distances) + 1) / len(sorted_distances) * 100
axes[1].plot(sorted_distances, cumulative, linewidth=2)
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Cumulative % of Test Cases')
axes[1].set_title('Cumulative Distance Distribution')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(50, color='red', linestyle='--', alpha=0.5, label='50th percentile')
axes[1].axhline(90, color='orange', linestyle='--', alpha=0.5, label='90th percentile')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print percentiles
print("\nDistance Percentiles:")
for p in [10, 25, 50, 75, 90, 95]:
    val = np.percentile(valid_comparisons['distance_km'], p)
    print(f"  {p}th percentile: {val:.2f} km")

### 6.8 Compare Model Geocodes vs. Manual Baseline

This section compares the T5 model's geocoded predictions against the **original manual baseline coordinates** that were looked up by humans directly in Google Maps (not re-geocoded through the API).

#### 6.8.1 Load Baseline Data

In [ ]:
# Load the original dataset with manual baseline coordinates
# This should match the dataset used to create the test set
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info.csv"
baseline_data = pd.read_csv(url, dtype=str)

# Convert latitude/longitude to float
baseline_data['baseline_lat'] = pd.to_numeric(baseline_data['latitude'], errors='coerce')
baseline_data['baseline_lon'] = pd.to_numeric(baseline_data['longitude'], errors='coerce')

print(f"✅ Loaded baseline data: {len(baseline_data)} total incidents")
print(f"   Manual coordinates available: {baseline_data['baseline_lat'].notna().sum()} incidents")

# Check if we can match by index or need to use incident_summary
if 'incident_summary' in test_predictions.columns and 'incident_summary' in baseline_data.columns:
    print("\n   Matching by incident_summary column...")
    # Merge geocoded results with baseline coordinates
    geocoded_with_baseline = geocoded_df.merge(
        baseline_data[['incident_summary', 'baseline_lat', 'baseline_lon']],
        left_on='input_text',
        right_on='incident_summary',
        how='left'
    )
else:
    print("\n   Warning: Cannot match by incident_summary. Using index-based match...")
    print("   This assumes test set maintains same order as original data.")
    # Fall back to matching by index (less reliable)
    geocoded_with_baseline = geocoded_df.copy()
    # Add baseline coordinates by index
    geocoded_with_baseline['baseline_lat'] = baseline_data['baseline_lat'].iloc[:len(geocoded_with_baseline)].values
    geocoded_with_baseline['baseline_lon'] = baseline_data['baseline_lon'].iloc[:len(geocoded_with_baseline)].values

print(f"\n✅ Matched {geocoded_with_baseline['baseline_lat'].notna().sum()} test cases with manual baseline coordinates")

#### 6.8.2 Calculate distance between model geocodes and manual baseline

In [ ]:
# Calculate distance between model geocodes and manual baseline
geocoded_with_baseline['distance_to_baseline_km'] = geocoded_with_baseline.apply(
    lambda row: haversine_distance(
        row['model_lat'], row['model_lon'],
        row['baseline_lat'], row['baseline_lon']
    ),
    axis=1
)

# Filter to cases where both model and baseline have coordinates
valid_baseline_comparisons = geocoded_with_baseline[
    geocoded_with_baseline['distance_to_baseline_km'].notna()
].copy()

print("="*80)
print("MODEL GEOCODES VS. MANUAL BASELINE COMPARISON")
print("="*80)
print(f"\nValid comparisons (both model and baseline available): {len(valid_baseline_comparisons)}/{len(geocoded_with_baseline)}")

if len(valid_baseline_comparisons) > 0:
    print(f"\nDistance Statistics (Model vs. Manual Baseline):")
    print(f"  Mean distance:   {valid_baseline_comparisons['distance_to_baseline_km'].mean():.2f} km")
    print(f"  Median distance: {valid_baseline_comparisons['distance_to_baseline_km'].median():.2f} km")
    print(f"  Min distance:    {valid_baseline_comparisons['distance_to_baseline_km'].min():.2f} km")
    print(f"  Max distance:    {valid_baseline_comparisons['distance_to_baseline_km'].max():.2f} km")
    print(f"  Std deviation:   {valid_baseline_comparisons['distance_to_baseline_km'].std():.2f} km")
    
    print(f"\nDistance Distribution (Practical Accuracy):")
    print(f"  Within 10 km:   {(valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Within 50 km:   {(valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Within 100 km:  {(valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Beyond 100 km:  {(valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    
    print("\nPercentiles:")
    for p in [10, 25, 50, 75, 90, 95]:
        val = np.percentile(valid_baseline_comparisons['distance_to_baseline_km'], p)
        print(f"  {p}th percentile: {val:.2f} km")
    
    print("\nSample comparisons:")
    display(valid_baseline_comparisons[['model_prediction', 'model_lat', 'model_lon', 'baseline_lat', 'baseline_lon', 'distance_to_baseline_km']].head(10))
else:
    print("\n⚠️  No valid comparisons available. Check data matching.")

#### 6.8.3 Save Results with Baseline Comparison

In [ ]:
# Save the results with baseline comparison
try:
    saved_path = save_dataframe_csv(geocoded_with_baseline, "location_extraction_geocoding_with_baseline.csv", task_name=TASK_NAME)
    print(f"✅ Results with baseline comparison saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_with_baseline.csv"
    geocoded_with_baseline.to_csv(output_path, index=False)
    print(f"✅ Results with baseline comparison saved to: {output_path}")

# Save summary metrics comparing to baseline
if len(valid_baseline_comparisons) > 0:
    baseline_summary = {
        'total_test_cases': len(geocoded_with_baseline),
        'valid_comparisons_to_baseline': len(valid_baseline_comparisons),
        'mean_distance_to_baseline_km': valid_baseline_comparisons['distance_to_baseline_km'].mean(),
        'median_distance_to_baseline_km': valid_baseline_comparisons['distance_to_baseline_km'].median(),
        'within_10km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum(),
        'within_50km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum(),
        'within_100km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum(),
        'beyond_100km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum(),
    }
    
    baseline_summary_df = pd.DataFrame([baseline_summary])
    
    try:
        saved_path = save_dataframe_csv(baseline_summary_df, "location_extraction_baseline_metrics.csv", task_name=TASK_NAME)
        print(f"✅ Baseline comparison metrics saved to: {saved_path}")
    except NameError:
        output_path = "./location_extraction_baseline_metrics.csv"
        baseline_summary_df.to_csv(output_path, index=False)
        print(f"✅ Baseline comparison metrics saved to: {output_path}")
    
    display(baseline_summary_df)

### 6.9 Accuracy by Geographic Specificity Level

Analyze how geocoding accuracy varies by the level of geographic detail extracted (village-level vs. district-level vs. state-level).

#### 6.9.1 Classify Geographic Level

In [ ]:
# Classify each prediction by geographic specificity level
def classify_geographic_level(prediction_text):
    """
    Classify the geographic specificity of a prediction.
    Returns: 'village', 'district', 'state', or 'none'
    """
    if not prediction_text or prediction_text.strip() == '':
        return 'none'
    
    parsed = parse_structured_location(prediction_text)
    
    # Most specific: Has village or other_locations
    if parsed['village'] or parsed['other_locations']:
        return 'village'
    # Medium: Has district but no village
    elif parsed['district']:
        return 'district'
    # Least specific: Has only state
    elif parsed['state']:
        return 'state'
    else:
        return 'none'

# Add geographic level classification to our results
if len(valid_baseline_comparisons) > 0:
    valid_baseline_comparisons['geographic_level'] = valid_baseline_comparisons['model_prediction'].apply(classify_geographic_level)
    
    print("="*80)
    print("ACCURACY BY GEOGRAPHIC SPECIFICITY LEVEL")
    print("="*80)
    
    # Count by level
    level_counts = valid_baseline_comparisons['geographic_level'].value_counts()
    print(f"\nDistribution of predictions by geographic level:")
    for level in ['village', 'district', 'state', 'none']:
        count = level_counts.get(level, 0)
        pct = (count / len(valid_baseline_comparisons) * 100) if len(valid_baseline_comparisons) > 0 else 0
        print(f"  {level.capitalize():12} level: {count:4d} ({pct:5.1f}%)")
    
    # Analyze accuracy by level
    print(f"\n{'='*80}")
    print("DISTANCE METRICS BY GEOGRAPHIC LEVEL")
    print(f"{'='*80}\n")
    
    for level in ['village', 'district', 'state']:
        level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
        
        if len(level_data) > 0:
            print(f"\n{level.upper()}-LEVEL PREDICTIONS (n={len(level_data)}):")
            print(f"  Mean distance:   {level_data['distance_to_baseline_km'].mean():.2f} km")
            print(f"  Median distance: {level_data['distance_to_baseline_km'].median():.2f} km")
            print(f"  Std deviation:   {level_data['distance_to_baseline_km'].std():.2f} km")
            print(f"  Within 10 km:    {(level_data['distance_to_baseline_km'] <= 10).sum()} ({(level_data['distance_to_baseline_km'] <= 10).sum()/len(level_data)*100:.1f}%)")
            print(f"  Within 50 km:    {(level_data['distance_to_baseline_km'] <= 50).sum()} ({(level_data['distance_to_baseline_km'] <= 50).sum()/len(level_data)*100:.1f}%)")
            print(f"  Within 100 km:   {(level_data['distance_to_baseline_km'] <= 100).sum()} ({(level_data['distance_to_baseline_km'] <= 100).sum()/len(level_data)*100:.1f}%)")
        else:
            print(f"\n{level.upper()}-LEVEL: No predictions at this level")
    
else:
    print("⚠️  No valid baseline comparisons available for stratification analysis")

#### 6.9.2 Visualize Accuracy by Geographic Level

In [ ]:
# Visualize accuracy by geographic level
if len(valid_baseline_comparisons) > 0 and 'geographic_level' in valid_baseline_comparisons.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Box plot of distances by level
    level_order = ['village', 'district', 'state']
    data_by_level = [
        valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]['distance_to_baseline_km'].dropna()
        for level in level_order
    ]
    
    bp = axes[0].boxplot(data_by_level, labels=[l.capitalize() for l in level_order], 
                         patch_artist=True, showmeans=True)
    axes[0].set_ylabel('Distance to Baseline (km)')
    axes[0].set_xlabel('Geographic Specificity Level')
    axes[0].set_title('Distance Distribution by Geographic Level')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Color the boxes
    colors = ['lightgreen', 'lightblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    # Bar chart of accuracy rates (% within thresholds)
    thresholds = [10, 50, 100]
    x = np.arange(len(level_order))
    width = 0.25
    
    for i, threshold in enumerate(thresholds):
        rates = []
        for level in level_order:
            level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
            if len(level_data) > 0:
                rate = (level_data['distance_to_baseline_km'] <= threshold).sum() / len(level_data) * 100
                rates.append(rate)
            else:
                rates.append(0)
        
        axes[1].bar(x + i*width, rates, width, label=f'Within {threshold} km')
    
    axes[1].set_xlabel('Geographic Specificity Level')
    axes[1].set_ylabel('% of Predictions')
    axes[1].set_title('Accuracy Rate by Geographic Level')
    axes[1].set_xticks(x + width)
    axes[1].set_xticklabels([l.capitalize() for l in level_order])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*80)
    print("KEY INSIGHT:")
    print("="*80)
    print("Village-level predictions should be more accurate (lower median distance)")
    print("District-level predictions will have higher error (using district centroid)")
    print("State-level predictions will have highest error (using state centroid)")
else:
    print("⚠️  Cannot create visualizations - no data with geographic levels")

#### 6.9.3 Save Accuracy by Geographic Level

In [ ]:
# Save stratified results
if len(valid_baseline_comparisons) > 0 and 'geographic_level' in valid_baseline_comparisons.columns:
    # Create summary by level
    level_summary = []
    for level in ['village', 'district', 'state']:
        level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
        if len(level_data) > 0:
            level_summary.append({
                'geographic_level': level,
                'count': len(level_data),
                'mean_distance_km': level_data['distance_to_baseline_km'].mean(),
                'median_distance_km': level_data['distance_to_baseline_km'].median(),
                'std_distance_km': level_data['distance_to_baseline_km'].std(),
                'pct_within_10km': (level_data['distance_to_baseline_km'] <= 10).sum() / len(level_data) * 100,
                'pct_within_50km': (level_data['distance_to_baseline_km'] <= 50).sum() / len(level_data) * 100,
                'pct_within_100km': (level_data['distance_to_baseline_km'] <= 100).sum() / len(level_data) * 100,
            })
    
    level_summary_df = pd.DataFrame(level_summary)
    
    try:
        saved_path = save_dataframe_csv(level_summary_df, "location_extraction_accuracy_by_level.csv", task_name=TASK_NAME)
        print(f"✅ Accuracy by geographic level saved to: {saved_path}")
    except NameError:
        output_path = "./location_extraction_accuracy_by_level.csv"
        level_summary_df.to_csv(output_path, index=False)
        print(f"✅ Accuracy by geographic level saved to: {output_path}")
    
    display(level_summary_df)